Let's start with roughly analysing the tables printing some charateristics.

In [130]:
import pandas as pd

prices1 = pd.read_csv('../data/prices1.csv')
prices2 = pd.read_csv('../data/prices2.csv')
weather = pd.read_csv('../data/weather.csv')

print('prices1 details')
print(f'Rows and Columns:\n{prices1.shape}\n')
print(f'Column Names:\n{prices1.columns.tolist()}\n')
print(f'Data types:\n{prices1.dtypes}\n')
print(f'Samples:\n{prices1.sample(5)}')
print('-----------------------------------------------------------------------------------------------\n')

print('prices2 details')
print(f'Rows and Columns:\n{prices2.shape}\n')
print(f'Column Names:\n{prices2.columns.tolist()}\n')
print(f'Data types:\n{prices2.dtypes}\n')
print(f'Samples:\n{prices2.sample(5)}')
print('-----------------------------------------------------------------------------------------------\n')

print('weather details')
print(f'Rows and Columns:\n{weather.shape}\n')
print(f'Column Names:\n{weather.columns.tolist()}\n')
print(f'Data types:\n{weather.dtypes}\n')
print(f'Samples:\n{weather.sample(5)}')
print('-----------------------------------------------------------------------------------------------')

prices1 details
Rows and Columns:
(16703, 11)

Column Names:
['delivery_date', 'id', 'business_date', 'time', 'time_interval', 'fixing_i_price', 'fixing_i_volume', 'fixing_ii_price', 'fixing_ii_volume', 'continous_trading_WAvg_price', 'continous_trading_volume']

Data types:
delivery_date                       str
id                                int64
business_date                       str
time                                str
time_interval                       str
fixing_i_price                  float64
fixing_i_volume                 float64
fixing_ii_price                 float64
fixing_ii_volume                float64
continous_trading_WAvg_price    float64
continous_trading_volume        float64
dtype: object

Samples:
                   delivery_date     id business_date      time time_interval  \
12750  2025-06-15 00:00:00+00:00   2693    2025-06-14  21:00:00         21-22   
1129   2024-02-17 00:00:00+00:00  11840    2024-02-16  23:00:00         23-24   
6576   2024-10-01

prices1 columns: ['delivery_date', 'id', 'business_date', 'time', 'time_interval', 'fixing_i_price', 'fixing_i_volume', 'fixing_ii_price', 'fixing_ii_volume', 'continous_trading_WAvg_price', 'continous_trading_volume']

prices2 columns: ['delivery_date', 'id', 'business_date', 'time', 'time_interval', 'fixing_i_price', 'fixing_i_volume', 'fixing_ii_price', 'fixing_ii_volume', 'continous_trading_WAvg_price', 'continous_trading_volume', 'fixing_ii_buy_volume', 'fixing_ii_sell_volume', 'total_listings_WAvg_price', 'total_listings_buy_volume', 'total_listings_max_price', 'total_listings_min_price', 'total_listings_sell_volume', 'total_listings_volume']

One quick thing we can do is to remove columns that we don't need in prices*.csv leaving only the temporal data and the fixing_i_price and rename the weather columns in English to have consistency with the others.

In [131]:
prices1.drop(columns=['fixing_i_volume', 'fixing_ii_price', 'fixing_ii_volume', 'continous_trading_WAvg_price', 'continous_trading_volume'], inplace=True)
prices2.drop(columns=['fixing_i_volume', 'fixing_ii_price', 'fixing_ii_volume', 'continous_trading_WAvg_price', 'continous_trading_volume', 'fixing_ii_buy_volume', 'fixing_ii_sell_volume', 'total_listings_WAvg_price', 'total_listings_buy_volume', 'total_listings_max_price', 'total_listings_min_price', 'total_listings_sell_volume', 'total_listings_volume'], inplace=True)
weather.rename(columns={'czas': 'time', 'id': 'id', 'miasto': 'city', 'temperatura': 'temperature', 
                        'wilgotnosc': 'humidity', 'opady': 'precipitation', 'naslonecznienie': 'sunny', 
                        'predkosc_wiatru': 'wind_speed'}, inplace=True)
print(f'Columns left in prices1: {prices1.columns.tolist()}')
print(f'Columns left in prices2: {prices2.columns.tolist()}')
print(f'Renamed weather columns: {weather.columns.tolist()}')

Columns left in prices1: ['delivery_date', 'id', 'business_date', 'time', 'time_interval', 'fixing_i_price']
Columns left in prices2: ['delivery_date', 'id', 'business_date', 'time', 'time_interval', 'fixing_i_price']
Renamed weather columns: ['time', 'id', 'city', 'temperature', 'humidity', 'precipitation', 'sunny', 'wind_speed']


Now let's work a bit with the time fields.

For the price1 table we have the following fields:
    delivery_date                       str (interpretation-> when electricity is delivered)
    business_date                       str (interpretation-> when the market auction happened)
    time                                str (interpretation-> when electricity is acutioned/delivered)
    time_interval                       str (interpretation-> the time interval in which the electricity has been auctioned)

The assumption is that people trade electricity on day x (business_date) and then the same electricity is delivered on day x+1 (delivery_date). For the time field I assume that electricity price is assigned in that hour and than delivered in the same hour.

For the price2 table we have the following fields:
    delivery_date                       str
    business_date                       str
    time                                str
    time_interval                       str

And for weather we have only: time

In [132]:
print("Time values in prices1:")
print(f'{prices1[["delivery_date", "business_date", "time", "time_interval"]].head(20)}\n')
print(f'{prices1["time"].unique()}\n')
print(f'{prices1["time_interval"].unique()}\n')
print("-----------------------------------------------------------------------------------------------\n")

print("Time values in prices2:")
print(f'{prices2[["delivery_date", "business_date", "time", "time_interval"]].head(20)}\n')
print(f'{prices2["time"].unique()}\n')
print(f'{prices2["time_interval"].unique()}\n')

Time values in prices1:
                delivery_date business_date      time time_interval
0   2024-01-01 00:00:00+00:00    2023-12-31  19:00:00         19-20
1   2024-01-01 00:00:00+00:00    2023-12-31  18:00:00         18-19
2   2024-01-01 00:00:00+00:00    2023-12-31  23:00:00         23-24
3   2024-01-01 00:00:00+00:00    2023-12-31  17:00:00         17-18
4   2024-01-01 00:00:00+00:00    2023-12-31  16:00:00         16-17
5   2024-01-01 00:00:00+00:00    2023-12-31  15:00:00         15-16
6   2024-01-01 00:00:00+00:00    2023-12-31  14:00:00         14-15
7   2024-01-01 00:00:00+00:00    2023-12-31  13:00:00         13-14
8   2024-01-01 00:00:00+00:00    2023-12-31  12:00:00         12-13
9   2024-01-01 00:00:00+00:00    2023-12-31  11:00:00         11-12
10  2024-01-01 00:00:00+00:00    2023-12-31  10:00:00         10-11
11  2024-01-01 00:00:00+00:00    2023-12-31  09:00:00          9-10
12  2024-01-01 00:00:00+00:00    2023-12-31  20:00:00         20-21
13  2024-01-01 00:00:00+

We can notice that time_interval is just a label for time so we can think to combine the time with delivery date to merge those informations into one.

In [133]:
# With dt.normalize() we set the time to 00:00:00, then we add the right time component using pd.to_timedelta()
prices1["delivery_timestamp"] = pd.to_datetime(prices1["delivery_date"].astype(str)).dt.normalize() + pd.to_timedelta(prices1["time"].astype(str))
prices2["delivery_timestamp"] = pd.to_datetime(prices2["delivery_date"].astype(str)).dt.normalize() + pd.to_timedelta(prices2["time"].astype(str))
weather["time"] = pd.to_datetime(weather["time"])

print(prices1[["delivery_date", "business_date", "time", "time_interval", "delivery_timestamp"]].head())
print("-----------------------------------------------------------------------------------------------\n")
print(prices2[["delivery_date", "business_date", "time", "time_interval", "delivery_timestamp"]].head())
print("-----------------------------------------------------------------------------------------------\n")
print(weather[["time"]].head())


               delivery_date business_date      time time_interval  \
0  2024-01-01 00:00:00+00:00    2023-12-31  19:00:00         19-20   
1  2024-01-01 00:00:00+00:00    2023-12-31  18:00:00         18-19   
2  2024-01-01 00:00:00+00:00    2023-12-31  23:00:00         23-24   
3  2024-01-01 00:00:00+00:00    2023-12-31  17:00:00         17-18   
4  2024-01-01 00:00:00+00:00    2023-12-31  16:00:00         16-17   

         delivery_timestamp  
0 2024-01-01 19:00:00+00:00  
1 2024-01-01 18:00:00+00:00  
2 2024-01-01 23:00:00+00:00  
3 2024-01-01 17:00:00+00:00  
4 2024-01-01 16:00:00+00:00  
-----------------------------------------------------------------------------------------------

               delivery_date business_date      time time_interval  \
0  2025-09-01 00:00:00+00:00    2025-08-31  01:00:00  01-09-25_H01   
1  2025-09-01 00:00:00+00:00    2025-08-31  02:00:00  01-09-25_H02   
2  2025-09-01 00:00:00+00:00    2025-08-31  03:00:00  01-09-25_H03   
3  2025-09-01 00:00:00

Let's look fro some duplicate presence and drop them if necessary:

In [134]:
print(f'Duplicates timestamps in prices1: {prices1["delivery_timestamp"].duplicated().sum()}')
# Here we show an example of duplicated timestamp in prices1.
print(prices1[prices1["delivery_timestamp"].duplicated(keep=False) == True].sort_values(["delivery_timestamp"]).head(10))
prices1.drop_duplicates(subset=["delivery_timestamp"], inplace=True)
print('-----------------------------------------------------------------------------------------------\n')

print(f'Duplicates timestamps in prices2: {prices2["delivery_timestamp"].duplicated().sum()}')
# Here we show an example of duplicated timestamp in prices2.
print(prices2[prices2["delivery_timestamp"].duplicated(keep=False) == True].sort_values(["delivery_timestamp"]).head(10))
#prices2.drop_duplicates(subset=["delivery_timestamp"], inplace=True)
print('-----------------------------------------------------------------------------------------------\n')

print(f'Duplicates timestamps in weather: {weather.duplicated(["time", "city"]).sum()}')

prices1.drop(columns=["delivery_date", "time", "time_interval"], inplace=True)
prices2.drop(columns=["delivery_date", "time", "time_interval"], inplace=True)

Duplicates timestamps in prices1: 24
                   delivery_date     id business_date      time time_interval  \
16606  2025-11-24 00:00:00+00:00  24008    2025-11-23  00:00:00         00-01   
16628  2025-11-24 00:00:00+00:00  23960    2025-11-23  00:00:00         00-01   
16627  2025-11-24 00:00:00+00:00  23961    2025-11-23  01:00:00         01-02   
16604  2025-11-24 00:00:00+00:00  24009    2025-11-23  01:00:00         01-02   
16587  2025-11-24 00:00:00+00:00  24010    2025-11-23  02:00:00         02-03   
16626  2025-11-24 00:00:00+00:00  23962    2025-11-23  02:00:00         02-03   
16590  2025-11-24 00:00:00+00:00  24011    2025-11-23  03:00:00         03-04   
16625  2025-11-24 00:00:00+00:00  23963    2025-11-23  03:00:00         03-04   
16589  2025-11-24 00:00:00+00:00  24012    2025-11-23  04:00:00         04-05   
16624  2025-11-24 00:00:00+00:00  23964    2025-11-23  04:00:00         04-05   

       fixing_i_price        delivery_timestamp  
16606          445.86

Here we can see a sumup of what we have been left with:

In [135]:
print(prices1.head())
print("-----------------------------------------------------------------------------------------------\n")
print(prices2.head())
print("-----------------------------------------------------------------------------------------------\n")
print(weather.head())   

      id business_date  fixing_i_price        delivery_timestamp
0  10708    2023-12-31          330.00 2024-01-01 19:00:00+00:00
1  10707    2023-12-31          336.11 2024-01-01 18:00:00+00:00
2  10712    2023-12-31          250.00 2024-01-01 23:00:00+00:00
3  10706    2023-12-31          340.00 2024-01-01 17:00:00+00:00
4  10705    2023-12-31          330.00 2024-01-01 16:00:00+00:00
-----------------------------------------------------------------------------------------------

   id business_date  fixing_i_price        delivery_timestamp
0   1    2025-08-31          398.99 2025-09-01 01:00:00+00:00
1   2    2025-08-31          383.00 2025-09-01 02:00:00+00:00
2   3    2025-08-31          368.54 2025-09-01 03:00:00+00:00
3   4    2025-08-31          361.98 2025-09-01 04:00:00+00:00
4   5    2025-08-31          369.00 2025-09-01 05:00:00+00:00
-----------------------------------------------------------------------------------------------

                       time      id       ci

We reached the point in which prices1 and prices2 have the same columns. Now we can union them resetting the index and later aggregate the values by delivery_timestamp so that we manage the overlapping time period between the tables.

After that we can join the weather data.

In [140]:
concat_prices = pd.concat([prices1, prices2], ignore_index=True)
print(f'Sample of rows to aggregate:\n{concat_prices[concat_prices.duplicated(subset=["delivery_timestamp"], keep=False) == True].sort_values(["delivery_timestamp"]).head(10)}')

concat_prices_agg = concat_prices.groupby("delivery_timestamp").agg({
    "business_date": "first",
    "fixing_i_price": "mean"
}).reset_index()

print(f'Number of duplicates: {concat_prices_agg.duplicated(subset=["delivery_timestamp"]).sum()}')

price_weather = pd.merge(left=concat_prices_agg, 
                         right=weather, 
                         left_on="delivery_timestamp", 
                         right_on="time", 
                         how="left").drop(columns=["time", "id"])
print(len(price_weather["city"].unique().tolist()))

Sample of rows to aggregate:
          id business_date  fixing_i_price        delivery_timestamp
16702     24    2025-08-31          422.10 2025-09-01 00:00:00+00:00
14628  21992    2025-08-31          398.99 2025-09-01 00:00:00+00:00
16679      1    2025-08-31          398.99 2025-09-01 01:00:00+00:00
14629  21993    2025-08-31          383.00 2025-09-01 01:00:00+00:00
14630  21994    2025-08-31          368.54 2025-09-01 02:00:00+00:00
16680      2    2025-08-31          383.00 2025-09-01 02:00:00+00:00
16681      3    2025-08-31          368.54 2025-09-01 03:00:00+00:00
14631  21995    2025-08-31          361.98 2025-09-01 03:00:00+00:00
16682      4    2025-08-31          361.98 2025-09-01 04:00:00+00:00
14632  21996    2025-08-31          369.00 2025-09-01 04:00:00+00:00
Number of duplicates: 0
19


The idea is to keep all cities weather condition to understand also the weather of which city influence the most the price.
In order to do that we can consider to pivot the city and weather condition.
Then add the time features, and lag price.

In [145]:
city_features = price_weather.pivot_table(
    index=["delivery_timestamp", "business_date", "fixing_i_price"],
    columns="city",
    values=["temperature", "humidity", "precipitation", "sunny", "wind_speed"]
)

city_features.columns = [
    f"{metric}_{city}"
    for metric, city in city_features.columns
]

city_features = city_features.reset_index()

city_features["hour"] = city_features["delivery_timestamp"].dt.hour
city_features["dow"] = city_features["delivery_timestamp"].dt.dayofweek
city_features["month"] = city_features["delivery_timestamp"].dt.month

city_features = city_features.sort_values("delivery_timestamp")

city_features["lag_24"] = city_features["fixing_i_price"].shift(24)
city_features["lag_48"] = city_features["fixing_i_price"].shift(48)
city_features["roll_mean_24"] = (
    city_features["fixing_i_price"]
    .shift(1)
    .rolling(24)
    .mean()
)

agg = price_weather.groupby("delivery_timestamp").agg({
    "temperature": ["mean", "min", "max", "std"],
    "wind_speed": ["mean", "max"],
    "humidity": ["mean"],
    "precipitation": ["sum"],
    "sunny": ["sum"]
})

agg.columns = [
    f"{col}_{stat}" for col, stat in agg.columns
]

agg = agg.reset_index()

final = city_features.merge(agg, on="delivery_timestamp",)

print(final.head(25))

          delivery_timestamp business_date  fixing_i_price  \
0  2024-01-01 00:00:00+00:00    2023-12-31          236.11   
1  2024-01-01 01:00:00+00:00    2023-12-31          236.10   
2  2024-01-01 02:00:00+00:00    2023-12-31          212.63   
3  2024-01-01 03:00:00+00:00    2023-12-31          212.00   
4  2024-01-01 04:00:00+00:00    2023-12-31          212.00   
5  2024-01-01 05:00:00+00:00    2023-12-31          236.10   
6  2024-01-01 06:00:00+00:00    2023-12-31          247.19   
7  2024-01-01 07:00:00+00:00    2023-12-31          250.00   
8  2024-01-01 08:00:00+00:00    2023-12-31          250.00   
9  2024-01-01 09:00:00+00:00    2023-12-31          282.00   
10 2024-01-01 10:00:00+00:00    2023-12-31          324.21   
11 2024-01-01 11:00:00+00:00    2023-12-31          341.82   
12 2024-01-01 12:00:00+00:00    2023-12-31          359.42   
13 2024-01-01 13:00:00+00:00    2023-12-31          371.97   
14 2024-01-01 14:00:00+00:00    2023-12-31          359.00   
15 2024-

To capture both local heterogeneity and national weather regimes, I combined city-level weather variables with aggregated country-wide statistics.

In this way during the analysis will be possible